<a href="https://colab.research.google.com/github/abod73/Ai_translation/blob/main/notebook18Abody3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📁 Project Structure: `config.py`

In [10]:
%%writefile config.py
import os
frfromom google.colab import userdata
from
# Telegram Bot API Credentials (Get from @BotFather and my.telegram.org)
BOT_TOKEN = userdata.get('BOT_TOKEN') # Your bot token
API_ID = userdata.get('API_ID') # Your API ID
API_HASH = userdata.get('API_HASH') # Your API Hash

# Video Download Settings
RESOLUTION = "720p" # Preferred video resolution (e.g., "1080p", "720p", "360p")

# Folder Paths
DOWNLOAD_PATH = "./downloads"
OUTPUT_PATH = "./output"
TEMP_AUDIO_PATH = os.path.join(DOWNLOAD_PATH, "temp_audio.wav")
TEMP_VIDEO_PATH = os.path.join(DOWNLOAD_PATH, "temp_video.mp4")

# Ensure directories exist
os.makedirs(DOWNLOAD_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Speech-to-Text Settings
FASTER_WHISPER_MODEL = "large-v3"
FASTER_WHISPER_VAD = True

# Translation Settings
SOURCE_LANG = "tr" # Source language (Turkish)
TARGET_LANG = "ar" # Target language (Arabic)
NAMES_FILE = "names.json" # File for protecting names during translation

# Qwen Model Settings (for translation)
QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

QWEN_GENERATION_CONFIG = {
    "max_new_tokens": 256,
    "temperature": 0.3,
    "top_p": 0.9,
    "do_sample": True,
    "repetition_penalty": 1.1,
}

# Qwen Chat Template - placeholder, will be filled dynamically by tokenizer
# For Qwen models, this is typically handled by tokenizer.apply_chat_template
# This variable is mainly for reference if a custom template string was needed.
# Example: QWEN_CHAT_TEMPLATE = "<|im_start|>system\nYou are a professional translator...<|im_end|>\n<|im_start|>user\n{user_message}<|im_end|>\n<|im_start|>assistant\n"


Writing config.py


## 📁 Project Structure: `utils.py`

In [8]:
%%writefile utilutilsutilss.py
import re
import asyncio
import validators
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def clean_filename(text: str) -> str:
    """
    دالة لتنظيف اسم الملف من الأحرف غير الصالحة.
    """
    # Remove invalid characters
    text = re.sub(r'[<>:"/\\|?*]', '', text)
    # Replace spaces with underscores
    text = text.replace(' ', '_')
    # Trim leading/trailing whitespace
    text = text.strip()
    # Limit length to avoid issues
    if len(text) > 100:
        text = text[:100]
    return text

def format_time(seconds: float) -> str:
    """
    دالة لتنسيق الوقت من الثواني إلى H:MM:SS.
    """
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h:01d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def is_valid_url(url: str) -> bool:
    """
    دالة للتحقق مما إذا كان الرابط المدخل صحيحًا.
    """
    try:
        return validators.url(url)
    except validators.ValidationFailure:
        return False

def merge_segments(segments, max_gap=0.5):
    """
    دالة لدمج المقاطع النصية المتقاربة زمنياً أو لتشكيل جملة كاملة قبل الترجمة.

    Args:
        segments (list): قائمة من القواميس تحتوي على 'start', 'end', 'text'.
        max_gap (float): الحد الأقصى للفجوة الزمنية (بالثواني) لدمج مقطعين.

    Returns:
        list: قائمة بالمقاطع المدمجة.
    """
    if not segments:
        return []

    merged = []
    current_segment = None

    for segment in segments:
        if current_segment is None:
            current_segment = {'start': segment['start'], 'end': segment['end'], 'text': segment['text']}
        else:
            # Check for time gap
            time_gap = segment['start'] - current_segment['end']

            # Check if current segment text ends with a sentence terminator
            ends_sentence = re.search(r'[.?!]$', current_segment['text'].strip())

            if time_gap <= max_gap and not ends_sentence:
                # Merge segments if gap is small and current text doesn't end a sentence
                current_segment['end'] = segment['end']
                current_segment['text'] += " " + segment['text']
            else:
                # Otherwise, start a new segment
                merged.append(current_segment)
                current_segment = {'start': segment['start'], 'end': segment['end'], 'text': segment['text']}

    # Add the last segment if it exists
    if current_segment:
        merged.append(current_segment)

    # Secondary pass to ensure logical sentence boundaries for very short segments
    final_merged = []
    temp_sentence_buffer = None

    for seg in merged:
        if temp_sentence_buffer is None:
            temp_sentence_buffer = {'start': seg['start'], 'end': seg['end'], 'text': seg['text']}
        else:
            ends_sentence_buffer = re.search(r'[.?!]$', temp_sentence_buffer['text'].strip())
            # If buffer doesn't end a sentence, and the next segment is relatively short, merge them.
            # This tries to prevent very short, incomplete sentences from being sent to LLM.
            if not ends_sentence_buffer and len(seg['text'].split()) < 5: # Arbitrary small word count
                temp_sentence_buffer['end'] = seg['end']
                temp_sentence_buffer['text'] += " " + seg['text']
            else:
                final_merged.append(temp_sentence_buffer)
                temp_sentence_buffer = {'start': seg['start'], 'end': seg['end'], 'text': seg['text']}

    if temp_sentence_buffer:
        final_merged.append(temp_sentence_buffer)

    return final_merged

Writing utils.py


## 📁 Project Structure: `downloader.py`

In [5]:
%%writefile downloader.py
import yt_dlp
import os
import logging
from config import DOWNLOAD_PATH

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

async def download_video(url: str) -> dict:
    """
    Downloads a video from the given URL using yt-dlp.

    Args:
        url (str): The URL of the video to download.

    Returns:
        dict: A dictionary containing 'filepath', 'title', and 'duration' of the downloaded video,
              or None if download fails.
    """
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': os.path.join(DOWNLOAD_PATH, '%(title)s.%(ext)s'),
        'merge_output_format': 'mp4',
        'noplaylist': True,
        'cachedir': False,
        'quiet': True,
        'progress_hooks': [lambda d: None], # Suppress progress output
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = await asyncio.to_thread(ydl.extract_info, url, download=False)
            if not info:
                logging.error(f"Could not extract info for URL: {url}")
                return None

            filepath_template = ydl.prepare_filename(info)
            # Ensure mp4 extension for consistency, though yt-dlp usually handles it
            filepath = filepath_template.replace('.webm', '.mp4').replace('.mkv', '.mp4')
            title = info.get('title', 'unknown_video')
            duration = info.get('duration', 0)

            # Actual download
            await asyncio.to_thread(ydl.download, [url])
            logging.info(f"Downloaded: {filepath}")

            return {
                'filepath': filepath,
                'title': title,
                'duration': duration
            }
    except Exception as e:
        logging.error(f"Error downloading video from {url}: {e}", exc_info=True)
        return None

Writing downloader.py


## 🛠 Install Dependencies

In [3]:
%%writefile requirements.txt
transformers
accelerate
torch
optimum
auto-gptq
ctranslate
pyrogram==2.0.106
ytdl-patched
ffmpeg-python
validators
srt

Overwriting requirements.txt


In [4]:
!pip install faster-whisper
!pip install -r requirements.txt
!pip install python-dotenv # For local development, if needed, though Colab secrets are preferred.
!pip install huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.1/126.1 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Discarding https://files.pythonhosted.org/packages/90/e5/b22697903982284fe284568fb2663a2196694a8eee637f5cf4ccfe435a38/auto_gptq-0.7.1.tar.gz (from https://pypi.org/simple/auto-gptq/) (requires-python:>=3.8.0): Requested auto-gptq from https://files.pythonhosted.org/packages/90/e5/b22697903982284fe284568fb2663a2196694a8eee637f5cf4ccfe435a38/auto_gptq-0.7.1.tar.gz (from -r requirements.txt (line 5)) has inconsistent version: expected '0.7.1', but metadata has '0.7.1+cu1281'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.6/124.6 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Discarding https://files.pythonhosted.org/packages/34/71/c3e73cf17681f6ff4754ef8f4cb8b67af3def230fc8711eac1250bbd78d5/auto_gptq-0.7.0.tar.gz (from https://pypi.org/simple/auto-gptq/) (requires-python:>=3.8.0): Requested auto-gptq from https://files.pythonhosted.org/packages/34/7

In [ ]:
%%writefile requirements.txt
transformers
accelerate
torch
optimum
auto-gptq
ctranslate
https://github.com/m-bain/whisperX/releases/download/v3.1.1/faster_whisper-0.12.0-cp310-cp310-linux_x86_64.whl
pyrogram==2.0.106
ytdl-patched
ffmpeg-python
validators
srt


In [ ]:
!pip install -r requirements.txt
!pip install python-dotenv # For local development, if needed, though Colab secrets are preferred.
!pip install huggingface_hub

## 📁 Project Structure: `names.json`

In [6]:
%%writefile names.json
{
    "Ayşe": "عائشة",
    "Fatma": "فاطمة",
    "Mehmet": "محمد",
    "Ali": "علي",
    "Zeynep": "زينب",
    "Mustafa": "مصطفى",
    "Emine": "أمينة",
    "Hüseyin": "حسين",
    "Hatice": "خديجة",
    "Ahmet": "أحمد",
    "Selin": "سيلين",
    "Deniz": "دينيز",
    "Can": "جان",
    "Elif": "إليف",
    "Burak": "براق",
    "Aslı": "أسلي",
    "Kemal": "كمال",
    "Leyla": "ليلى",
    "Murat": "مراد",
    "Pelin": "بيلين",
    "Kadir": "قادر",
    "Nihan": "نيهان",
    "Kerem": "كرم",
    "Eylül": "إيلول",
    "Cem": "جيم",
    "Melis": "مليس",
    "Aras": "أراس",
    "Hande": "هانده",
    "Tolga": "تولغا",
    "Bahar": "بهار",
    "Okan": "أوكان",
    "Gül": "غول",
    "Serkan": "سركان",
    "Eda": "إيدا",
    "Barış": "باريش",
    "Defne": "دفنة",
    "Sinan": "سنان",
    "Merve": "ميرفي",
    "Koray": "كوراي",
    "Nazlı": "نازلي",
    "Engin": "إنجين",
    "Aslıhan": "أصلي هان",
    "Onur": "أونور",
    "Figen": "فيجين",
    "Umut": "أوموت",
    "Ceyda": "جيداء",
    "Volkan": "فولكان",
    "Tuğçe": "توتشه",
    "Selim": "سليم",
    "Gizem": "جيزام",
    "Doğan": "دوغان",
    "Damla": "داملة",
    "Mert": "ميرت",
    "Buse": "بوسه",
    "Erdem": "إردم",
    "Esra": "إسراء",
    "Berkay": "بيركاي",
    "Cansu": "جانسو",
    "Eren": "إرين",
    "Didem": "ديدم",
    "Fatih": "فاتح",
    "Gözde": "غوزده",
    "Hakan": "هاكان",
    "Işıl": "إشيل",
    "İlker": "إيلكر",
    "Jale": "جالة",
    "Kaan": "كان",
    "Lale": "لاله",
    "Özlem": "أوزلم",
    "Pınar": "بينار",
    "Rüya": "رؤيا",
    "Seda": "سيدا",
    "Tuna": "تونا",
    "Utku": "أوتكو",
    "Vildan": "فيلدان",
    "Yasemin": "ياسمين",
    "Zerrin": "زرين"
}

Writing names.json


## 📁 Project Structure: `processor.py`

In [11]:
%%writefile processor.py
import os
import json
import torch
import asyncio
import logging
import srt
import subprocess
import re

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from faster_whisper import WhisperModel

from config import (
    DOWNLOAD_PATH, OUTPUT_PATH, TEMP_AUDIO_PATH, TEMP_VIDEO_PATH,
    FASTER_WHISPER_MODEL, FASTER_WHISPER_VAD, SOURCE_LANG, TARGET_LANG,
    NAMES_FILE, QWEN_MODEL_NAME, QWEN_GENERATION_CONFIG
)
from utils import merge_segments, format_time, clean_filename

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Global Model Loading (to avoid reloading for each request) ---

whisper_model = None
qwen_tokenizer = None
qwen_model = None

async def load_models():
    """
    دالة لتحميل نماذج Faster-Whisper و Qwen مرة واحدة.
    """
    global whisper_model, qwen_tokenizer, qwen_model
    if whisper_model is None:
        logging.info(f"Loading Faster-Whisper model: {FASTER_WHISPER_MODEL}")
        # Use "cuda" if a GPU is available, otherwise "cpu"
        device = "cuda" if torch.cuda.is_available() else "cpu"
        compute_type = "float16" if device == "cuda" else "int8"
        whisper_model = WhisperModel(FASTER_WHISPER_MODEL, device=device, compute_type=compute_type)
        logging.info("Faster-Whisper model loaded.")

    if qwen_tokenizer is None or qwen_model is None:
        logging.info(f"Loading Qwen model: {QWEN_MODEL_NAME}")
        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
        qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            trust_remote_code=True
        )
        qwen_model.eval() # Set model to evaluation mode
        logging.info("Qwen model loaded.")

# --- Name Protection Functions ---

def load_names_map():
    """
    دالة لتحميل خريطة الأسماء من ملف JSON.
    """
    try:
        with open(NAMES_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        logging.warning(f"Names file '{NAMES_FILE}' not found. Name protection will be skipped.")
        return {}

def protect_names(text: str, names_dict: dict) -> tuple[str, dict]:
    """
    دالة لحماية الأسماء التركية من الترجمة الخاطئة عن طريق استبدالها بعلامات مؤقتة.

    Args:
        text (str): النص الأصلي باللغة التركية.
        names_dict (dict): قاموس يحتوي على الأسماء التركية ومقابلها العربي.

    Returns:
        tuple: النص مع العلامات المؤقتة، وقاموس ربط العلامات بالأسماء التركية الأصلية.
    """
    protected_text = text
    mapping = {}
    for i, (tr_name, ar_name) in enumerate(names_dict.items()):
        # Use word boundaries to match whole names only
        pattern = r'\b' + re.escape(tr_name) + r'\b'
        if re.search(pattern, protected_text, re.IGNORECASE):
            placeholder = f"__NAME_{i}__"
            protected_text = re.sub(pattern, placeholder, protected_text, flags=re.IGNORECASE)
            mapping[placeholder] = ar_name # Store the Arabic name to restore later
    return protected_text, mapping

def restore_names(translated_text: str, mapping: dict) -> str:
    """
    دالة لاستعادة الأسماء العربية المحمية بعد عملية الترجمة.

    Args:
        translated_text (str): النص المترجم الذي يحتوي على العلامات المؤقتة.
        mapping (dict): قاموس ربط العلامات المؤقتة بالأسماء العربية الأصلية.

    Returns:
        str: النص المترجم بعد استعادة الأسماء.
    """
    restored_text = translated_text
    for placeholder, ar_name in mapping.items():
        restored_text = restored_text.replace(placeholder, ar_name)
    return restored_text

# --- Translation Prompt Construction ---

def build_qwen_prompt(
    text_to_translate: str,
    previous_context: list[str] = None,
    names_map: dict = None
) -> str:
    """
    دالة لبناء الـ Prompt الاحترافي لنموذج Qwen للترجمة.

    Args:
        text_to_translate (str): النص التركي المراد ترجمته.
        previous_context (list): قائمة بأخر جملتين عربيتين مترجمتين لتوفير السياق.
        names_map (dict): قاموس الأسماء لحماية الأسماء في الأمثلة إذا لزم الأمر.

    Returns:
        str: الـ Prompt الكامل لـ Qwen.
    """
    system_message = (
        "أنت مترجم محترف ومتخصص في دبلجة وترجمة المسلسلات التركية إلى العربية. "
        "مهمتك هي ترجمة النص التركي إلى عربية طبيعية، سلسة، وتحافظ على روح الحوار. "
        "إليك القواعد الصارمة التي يجب اتباعها:
        "1. لا تترجم حرفياً، بل ترجم المعنى والسياق.
        "2. حافظ على نبرة الحوار (رسمي، عاطفي، غاضب، أو عامي خفيف حسب السياق).
        "3. صحح الضمائر (هو/هي) بناءً على سياق الجملة السابقة.
        "4. لا تترجم أسماء الأشخاص أو الأماكن.
        "5. أخرج الترجمة النهائية فقط بدون أي شرح أو هوامش."
    )

    few_shot_examples = [
        {"tr": "Babam, ben bu şeyi anlamıyorum, neden bunu bana yapıyorsun?",
         "ar": "يا أبي، لم أعد أفهم شيئاً، لماذا تفعل بي هذا؟"},
        {"tr": "Seyran, hemen buraya gel.",
         "ar": "سيران، تعالي إلى هنا فوراً."}
    ]

    context_part = ""
    if previous_context:
        context_part = "\n\nالسياق السابق (مهم للضمائر):\n" + "\n".join(previous_context)

    full_prompt = f"""<|im_start|>system
{system_message}<|im_end|>
"
    for example in few_shot_examples:
        example_tr_protected, _ = protect_names(example['tr'], names_map) # Protect names in examples too
        full_prompt += f"<|im_start|>user\nالتركية: {example_tr_protected}<|im_end|>
<|im_start|>assistant\nالعربية: {example['ar']}<|im_end|>
"

    full_prompt += f"<|im_start|>user\nالتركية: {text_to_translate}{context_part}<|im_end|>
<|im_start|>assistant\nالعربية: "

    return full_prompt

# --- Core Translation Function ---

async def translate_segments(segments: list[dict]) -> list[dict]:
    """
    دالة لترجمة قائمة من المقاطع النصية باستخدام نموذج Qwen مع تطبيق تقنيات التحسين.

    Args:
        segments (list): قائمة من القواميس، كل قاموس يحتوي على 'start', 'end', 'text' باللغة التركية.

    Returns:
        list: قائمة بالمقاطع المترجمة، كل قاموس يحتوي على 'start', 'end', 'text' بالعربية.
    """
    await load_models() # Ensure models are loaded

    if not segments:
        return []

    names_dict = load_names_map()
    translated_segments = []
    previous_context = [] # Stores last 2 translated sentences for context

    # Step 1: Merge short segments first
    merged_segments = merge_segments(segments, max_gap=0.5)

    for i, seg in enumerate(merged_segments):
        original_text = seg['text']

        # Step 2: Protect names
        protected_text, name_mapping = protect_names(original_text, names_dict)

        # Step 3: Build professional prompt with previous context and few-shot examples
        prompt = build_qwen_prompt(protected_text, previous_context, names_dict)

        inputs = qwen_tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to(qwen_model.device)

        try:
            with torch.no_grad():
                outputs = await asyncio.to_thread(qwen_model.generate, **inputs, **QWEN_GENERATION_CONFIG)

            # Decode only the newly generated part
            generated_text = qwen_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

            # Clean up potential artifacts from generation if any
            translated_text_raw = generated_text.strip().split('\n')[0].replace('العربية: ', '').strip()

            # Step 5: Restore names
            final_translated_text = restore_names(translated_text_raw, name_mapping)

            translated_segments.append({
                'start': seg['start'],
                'end': seg['end'],
                'text': final_translated_text
            })

            # Update previous context (keep only the last 2 sentences)
            previous_context.append(final_translated_text)
            if len(previous_context) > 2:
                previous_context.pop(0)

            logging.info(f"Translated (seg {i+1}/{len(merged_segments)}):\nTR: {original_text}\nAR: {final_translated_text}")

        except Exception as e:
            logging.error(f"Error during Qwen translation for segment: {original_text} - {e}", exc_info=True)
            # Fallback to original text if translation fails (or a placeholder)
            translated_segments.append({'start': seg['start'], 'end': seg['end'], 'text': original_text})

    return translated_segments

# --- Video Processing Pipeline ---

async def extract_audio(video_path: str, audio_path: str):
    """
    دالة لاستخلاص الصوت من الفيديو باستخدام FFmpeg.
    """
    command = [
        'ffmpeg', '-y', '-i', video_path, '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', audio_path
    ]
    logging.info(f"Extracting audio: {' '.join(command)}")
    process = await asyncio.create_subprocess_exec(
        *command,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )
    stdout, stderr = await process.communicate()
    if process.returncode != 0:
        logging.error(f"FFmpeg audio extraction failed: {stderr.decode()}")
        raise Exception(f"FFmpeg audio extraction failed: {stderr.decode()}")
    logging.info(f"Audio extracted to {audio_path}")

async def generate_srt_file(translated_segments: list[dict], output_srt_path: str):
    """
    دالة لإنشاء ملف SRT من المقاطع المترجمة.
    """
    subs = []
    for i, segment in enumerate(translated_segments):
        start_time = srt.timedelta(seconds=segment['start'])
        end_time = srt.timedelta(seconds=segment['end'])
        subs.append(srt.Subtitle(index=i + 1, start=start_time, end=end_time, content=segment['text']))

    # Sort subtitles by start time to ensure correct order
    subs.sort(key=lambda x: x.start)

    with open(output_srt_path, 'w', encoding='utf-8') as f:
        f.write(srt.compose(subs))
    logging.info(f"SRT file generated: {output_srt_path}")
    return output_srt_path

async def hardsub_video(video_path: str, srt_path: str, output_video_path: str):
    """
    دالة لدمج ملف الترجمة (Hardsub) مع الفيديو باستخدام FFmpeg.
    تستخدم خطاً عربياً مدعوماً مع حدود وظل لضمان قابلية القراءة.
    """
    # Font settings for Arabic hardsubs. You might need to install 'Amiri' font in your environment
    # or replace with another suitable Arabic font available (e.g., 'Noto Sans Arabic').
    # For Colab, Noto Sans Arabic is usually available by default.
    font_path = '/usr/share/fonts/truetype/noto/NotoSansArabic-Regular.ttf' # Common path in Linux/Colab
    if not os.path.exists(font_path):
        logging.warning(f"Font not found at {font_path}. Falling back to a generic font or system default.")
        font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf" # Fallback to a common system font
        if not os.path.exists(font_path):
             logging.error(f"Fallback font not found at {font_path}. Hardsubbing might fail or use a default ugly font.")
             font_path = "sans-serif" # Last resort, let ffmpeg decide.

    # FFmpeg complex filter for subtitles
    # Parameters:
    # - force_style: FontName,FontSize,PrimaryColour,OutlineColour,BackColour,Outline,Shadow,MarginV
    # - PrimaryColour: &H00BBGGRR (Alpha Blue Green Red) e.g., &H0000FFFF for yellow
    # - OutlineColour: &H00BBGGRR e.g., &H00000000 for black
    # - BackColour: &H00BBGGRR e.g., &H80000000 for semi-transparent black background
    # - Outline: thickness of border
    # - Shadow: offset of shadow (x,y in pixels)
    sub_style = (
        f"Fontname={font_path}, "
        "FontSize=28, " # Adjust font size as needed
        "PrimaryColour=&H00FFFFFF, " # White color
        "OutlineColour=&H00000000, " # Black outline
        "BackColour=&H80000000, " # Semi-transparent black background
        "Outline=2, " # Outline thickness
        "Shadow=1, " # Shadow depth
        "MarginV=20, " # Vertical margin from bottom
        "Alignment=2" # Bottom center
    )

    command = [
        'ffmpeg', '-y', '-i', video_path, '-vf',
        f"subtitles={srt_path}:force_style='{sub_style}'",
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
        '-c:a', 'copy', output_video_path
    ]

    logging.info(f"Hardsubbing video: {' '.join(command)}")
    process = await asyncio.create_subprocess_exec(
        *command,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )
    stdout, stderr = await process.communicate()
    if process.returncode != 0:
        logging.error(f"FFmpeg hardsubbing failed: {stderr.decode()}")
        raise Exception(f"FFmpeg hardsubbing failed: {stderr.decode()}")
    logging.info(f"Hardsubbed video created: {output_video_path}")
    return output_video_path

async def clean_temp_files(*paths):
    """
    دالة لتنظيف الملفات المؤقتة.
    """
    for path in paths:
        if os.path.exists(path):
            os.remove(path)
            logging.info(f"Cleaned up temporary file: {path}")

async def process_video_pipeline(video_filepath: str, video_title: str) -> tuple[str, str]:
    """
    Pipeline متكاملة لمعالجة الفيديو: استخلاص الصوت، تحويل الكلام إلى نص، ترجمة،
    توليد ملف SRT، ودمج الترجمة مع الفيديو.

    Args:
        video_filepath (str): المسار الكامل لملف الفيديو الأصلي.
        video_title (str): عنوان الفيديو الأصلي.

    Returns:
        tuple: مسار الفيديو المترجم، مسار ملف SRT.

    Raises:
        Exception: إذا فشلت أي خطوة في Pipeline.
    """
    clean_video_title = clean_filename(video_title)
    output_video_filepath = os.path.join(OUTPUT_PATH, f"{clean_video_title}_translated.mp4")
    output_srt_filepath = os.path.join(OUTPUT_PATH, f"{clean_video_title}.srt")

    try:
        # 1. Extract Audio
        logging.info("Starting audio extraction...")
        await extract_audio(video_filepath, TEMP_AUDIO_PATH)

        # 2. Speech-to-Text with Faster-Whisper
        logging.info("Starting Speech-to-Text with Faster-Whisper...")
        segments_generator, info = await asyncio.to_thread(whisper_model.transcribe,
                                                            TEMP_AUDIO_PATH,
                                                            language=SOURCE_LANG,
                                                            vad_filter=FASTER_WHISPER_VAD)

        segments = []
        for segment in segments_generator:
            segments.append({
                'start': segment.start,
                'end': segment.end,
                'text': segment.text
            })
        logging.info(f"Speech-to-Text completed. Found {len(segments)} segments.")

        if not segments:
            raise Exception("No speech segments detected in the video.")

        # 3. Translate Segments with Qwen
        logging.info("Starting translation with Qwen...")
        translated_segments = await translate_segments(segments)
        logging.info("Translation completed.")

        # 4. Generate SRT file
        logging.info("Generating SRT file...")
        await generate_srt_file(translated_segments, output_srt_filepath)

        # 5. Hardsub Video
        logging.info("Starting hardsubbing video...")
        await hardsub_video(video_filepath, output_srt_filepath, output_video_filepath)

        logging.info("Video processing pipeline completed successfully.")
        return output_video_filepath, output_srt_filepath

    except Exception as e:
        logging.error(f"Video processing pipeline failed: {e}", exc_info=True)
        raise

    finally:
        # Clean up temporary files
        await clean_temp_files(TEMP_AUDIO_PATH, video_filepath)


Writing processor.py


## 📁 Project Structure: `main.py`

In [12]:
%%writefile main.py
import os
import asyncio
import logging
import time

from pyrogram import Client, filters
from pyrogram.types import InlineKeyboardMarkup, InlineKeyboardButton, Message
from pyrogram.errors import FloodWait

from config import API_ID, API_HASH, BOT_TOKEN, DOWNLOAD_PATH, OUTPUT_PATH
from downloader import download_video
from processor import process_video_pipeline, load_models
from utils import is_valid_url, format_time

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Initialize Pyrogram Client
app = Client(
    "TurkishVideoTranslatorBot",
    api_id=API_ID,
    api_hash=API_HASH,
    bot_token=BOT_TOKEN
)

# --- Global State for tracking user requests ---
# In a single-user bot, we can use a simple dict for simplicity.
# For multi-user, this would require a proper database or more complex state management.
user_states = {}

# --- Command Handlers ---

@app.on_message(filters.command("start"))
async def start_command(client: Client, message: Message):
    """
    يعالج الأمر /start ويرسل رسالة ترحيب.
    """
    logging.info(f"Received /start command from {message.from_user.id}")
    await message.reply_text(
        "مرحباً بك! أنا بوت لترجمة الفيديوهات التركية إلى العربية.\n"\
        "فقط أرسل لي رابط فيديو (YouTube, Twitter, إلخ) وسأتولى الباقي."
    )

@app.on_message(filters.regex(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+') & filters.private)
async def url_handler(client: Client, message: Message):
    """
    يعالج الروابط المرسلة من قبل المستخدم.
    """
    url = message.text.strip()
    user_id = message.from_user.id
    logging.info(f"Received URL: {url} from user {user_id}")

    if not is_valid_url(url):
        await message.reply_text("الرابط غير صالح. يرجى إرسال رابط فيديو صحيح.")
        return

    # Check if user is already processing a video
    if user_id in user_states and user_states[user_id].get('processing', False):
        await message.reply_text("أنا مشغول بمعالجة طلبك السابق. يرجى الانتظار حتى ينتهي.")
        return

    # Set user state to processing
    user_states[user_id] = {'processing': True, 'url': url}

    status_message = await message.reply_text("جاري جلب معلومات الفيديو... ⏳")

    try:
        # Pre-load models asynchronously to reduce latency when needed later
        await load_models()

        video_info = await download_video(url)

        if video_info:
            title = video_info['title']
            duration = video_info['duration']

            user_states[user_id]['video_info'] = video_info

            await status_message.edit_text(
                f"**عنوان الفيديو:** `{title}`\n"\
                f"**المدة:** `{format_time(duration)}`\n\n"\
                "هل تريد بدء الترجمة؟",
                reply_markup=InlineKeyboardMarkup(
                    [[
                        InlineKeyboardButton("🚀 ابدأ الترجمة", callback_data=f"translate_start")
                    ]]
                )
            )
        else:
            await status_message.edit_text("عذراً، لم أتمكن من جلب معلومات الفيديو أو تحميله. تأكد من أن الرابط صحيح ومدعوم.")
            user_states.pop(user_id, None) # Clear user state

    except Exception as e:
        logging.error(f"Error in URL handler for {url}: {e}", exc_info=True)
        await status_message.edit_text("حدث خطأ أثناء جلب معلومات الفيديو. يرجى المحاولة مرة أخرى لاحقاً.")
        user_states.pop(user_id, None) # Clear user state

@app.on_callback_query(filters.regex("translate_start"))
async def start_translation_callback(client: Client, callback_query):
    """
    يعالج ضغط زر 'ابدأ الترجمة'.
    """
    user_id = callback_query.from_user.id
    logging.info(f"Received translate_start callback from user {user_id}")

    if user_id not in user_states or not user_states[user_id].get('processing', False):
        await callback_query.answer("خطأ: لا يوجد طلب ترجمة قيد الانتظار.", show_alert=True)
        await callback_query.message.edit_text("الطلب غير صالح أو انتهت صلاحيته. يرجى إرسال رابط فيديو جديد.")
        return

    video_info = user_states[user_id].get('video_info')
    if not video_info:
        await callback_query.answer("خطأ: لا توجد معلومات فيديو.", show_alert=True)
        await callback_query.message.edit_text("خطأ في معلومات الفيديو. يرجى المحاولة مرة أخرى.")
        user_states.pop(user_id, None) # Clear user state
        return

    await callback_query.answer("بدء الترجمة...", show_alert=False)
    initial_message = callback_query.message

    try:
        # Update status message
        await initial_message.edit_text("جاري استخلاص الصوت وتحويله إلى نص... 🎙")

        # Run the full processing pipeline
        translated_video_path, srt_path = await process_video_pipeline(video_info['filepath'], video_info['title'])

        await initial_message.edit_text("جاري إرسال الفيديو المترجم وملف الترجمة... 📤")

        # Send the translated video
        if os.path.exists(translated_video_path):
            await client.send_video(
                chat_id=initial_message.chat.id,
                video=translated_video_path,
                caption=f"تمت الترجمة بنجاح! \n**العنوان الأصلي:** {video_info['title']}"
            )
            logging.info(f"Sent translated video: {translated_video_path}")
        else:
            await initial_message.reply_text("عذراً، لم أتمكن من العثور على الفيديو المترجم.")

        # Send the SRT file
        if os.path.exists(srt_path):
            await client.send_document(
                chat_id=initial_message.chat.id,
                document=srt_path,
                caption="ملف الترجمة (SRT)"
            )
            logging.info(f"Sent SRT file: {srt_path}")
        else:
            await initial_message.reply_text("عذراً، لم أتمكن من العثور على ملف الترجمة.")

        await initial_message.edit_text("✅ اكتملت المعالجة!")

    except FloodWait as e:
        logging.warning(f"FloodWait error: {e.value} seconds")
        await initial_message.edit_text(f"تجاوزت حدود تيليجرام. يرجى المحاولة مرة أخرى بعد {e.value} ثوانٍ.")
        await asyncio.sleep(e.value) # Wait before clearing state or trying again
    except Exception as e:
        logging.error(f"Error in translation pipeline for user {user_id}: {e}", exc_info=True)
        await initial_message.edit_text("حدث خطأ أثناء معالجة الفيديو. يرجى المحاولة مرة أخرى لاحقاً.")
    finally:
        # Clean up all temporary files created during processing
        if user_id in user_states and 'video_info' in user_states[user_id]:
            # processor.py's pipeline cleans its own temp files, ensure main downloaded file is also cleaned.
            if os.path.exists(user_states[user_id]['video_info']['filepath']):
                os.remove(user_states[user_id]['video_info']['filepath'])
                logging.info(f"Cleaned up original downloaded video: {user_states[user_id]['video_info']['filepath']}")

            # Clean up the final output files if they exist and were successfully sent.
            # This part can be adjusted if you want to keep output files after sending.
            cleaned_title = user_states[user_id].get('video_info', {}).get('title', 'video')
            output_video_filepath = os.path.join(OUTPUT_PATH, f"{clean_filename(cleaned_title)}_translated.mp4")
            output_srt_filepath = os.path.join(OUTPUT_PATH, f"{clean_filename(cleaned_title)}.srt")

            if os.path.exists(output_video_filepath):
                os.remove(output_video_filepath)
                logging.info(f"Cleaned up output translated video: {output_video_filepath}")
            if os.path.exists(output_srt_filepath):
                os.remove(output_srt_filepath)
                logging.info(f"Cleaned up output SRT: {output_srt_filepath}")

        user_states.pop(user_id, None) # Clear user state after processing (or failure)

# --- Main entry point ---

async def main():
    logging.info("Starting bot...")
    # Ensure models are pre-loaded at bot startup
    await load_models()
    await app.start()
    logging.info("Bot started. Listening for messages...")
    # Keep the bot running indefinitely
    await idle() # Pyrogram's idle function to keep the bot running

if __name__ == "__main__":
    # In a real application, you might use a more robust way to run the async main
    # like asyncio.run(main()) but for Colab interactive environment, this might be simpler
    # if not already inside an event loop.
    # For Pyrogram's app.run(), it handles the event loop internally.
    app.run()


Writing main.py


## 📄 `README.md`

In [13]:
%%writefile README.md
# Personal Telegram Video Translator Bot (Turkish to Arabic)

This is a personal Telegram bot designed to translate Turkish videos to Arabic, focusing on natural and fluent translation suitable for dubbing.
It's built for single-user operation with a clean, lightweight, and direct codebase.

## 🛠 Technologies Used

- **Telegram Bot:** [Pyrogram](https://pyrogram.org/) (Async)
- **Video Download:** [yt-dlp](https://github.com/yt-dlp/yt-dlp)
- **Speech-to-Text:** [Faster-Whisper](https://github.com/guillaumekln/faster-whisper) (large-v3) with VAD
- **Translation:** [Qwen/Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) (for core translation and refinement)
- **Video Processing:** [FFmpeg](https://ffmpeg.org/) (for hardsubbing with clear Arabic font)
- **Utilities:** `AsyncIO`, `logging`, `transformers`

## 📂 Project Structure

The project is organized into 5 main files:

1.  `config.py`: Stores static environment variables (BOT_TOKEN, API_ID, API_HASH, 720p quality, folder paths, Qwen settings).
2.  `utils.py`: Contains helper functions (filename cleaning, time formatting, URL validation, sentence merging).
3.  `downloader.py`: Simple function to download videos via `yt-dlp`, returning path and duration.
4.  `processor.py`: The core pipeline for audio extraction, STT, translation (with name protection and advanced prompting), SRT generation, and hardsubbing.
5.  `main.py`: Entry point for the bot, sets up the Pyrogram client, and handles incoming messages and callbacks.

## ✨ Key Translation Features (in `processor.py`)

To achieve high-quality, natural translation, the bot implements the following:

1.  **Sentence Merging:** Short segments from Faster-Whisper are merged based on time gaps (`< 0.5s`) and sentence termination to form more complete and meaningful sentences for the LLM.
2.  **Name Protection:** Uses `names.json` to identify and protect Turkish names, replacing them with temporary placeholders before translation and restoring them afterwards.
3.  **Advanced Translation Prompt:** Qwen is prompted with a detailed system role, strict translation rules (context, tone, pronoun correction, no literal translation), few-shot examples, and previous translated context (last two sentences) to ensure coherence and natural flow.
4.  **Qwen2.5-3B-Instruct Usage:** The model is loaded and used with optimized generation parameters for best translation quality.

## 🚀 How to Run the Bot

### 1. Setup API Keys and Tokens

-   **Telegram Bot Token:** Obtain from [@BotFather](https://t.me/BotFather) on Telegram.
-   **Telegram API ID & API Hash:** Get these from [my.telegram.org](https://my.telegram.org/apps).

### 2. Colab/Kaggle Environment (Recommended)

1.  **Open the Notebook:** Upload this notebook to Google Colab or Kaggle.
2.  **Add Secrets:**
    -   In the left panel of Colab, find the "🔑 Secrets" icon.
    -   Add three new secrets with the exact names:
        -   `BOT_TOKEN`
        -   `API_ID`
        -   `API_HASH`
    -   Paste your respective values into these secrets.
    -   Ensure "Notebook access" is enabled for these secrets.
3.  **Install Dependencies:** Run the cell containing `!pip install -r requirements.txt` and the other `pip install` commands.
4.  **Run Files:** The notebook creates `config.py`, `utils.py`, `downloader.py`, `processor.py`, and `main.py` in the Colab environment.
5.  **Start the Bot:** Run the cell that imports and calls `app.run()` (usually the last Python cell in `main.py` if structured as provided).

### 3. Local Environment

1.  **Clone the Repository:** (If this was a repository)
    ```bash
    git clone <repository_url>
    cd <repository_name>
    ```
2.  **Create `config.py`:** Manually create a `config.py` file with your credentials:
    ```python
    # config.py
    import os
    BOT_TOKEN = "YOUR_BOT_TOKEN"
    API_ID = YOUR_API_ID
    API_HASH = "YOUR_API_HASH"

    # Other configurations from the notebook's config.py...
    DOWNLOAD_PATH = "./downloads"
    OUTPUT_PATH = "./output"
    TEMP_AUDIO_PATH = os.path.join(DOWNLOAD_PATH, "temp_audio.wav")
    TEMP_VIDEO_PATH = os.path.join(DOWNLOAD_PATH, "temp_video.mp4")
    os.makedirs(DOWNLOAD_PATH, exist_ok=True)
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    FASTER_WHISPER_MODEL = "large-v3"
    FASTER_WHISPER_VAD = True
    SOURCE_LANG = "tr"
    TARGET_LANG = "ar"
    NAMES_FILE = "names.json"
    QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
    QWEN_GENERATION_CONFIG = {
        "max_new_tokens": 256,
        "temperature": 0.3,
        "top_p": 0.9,
        "do_sample": True,
        "repetition_penalty": 1.1,
    }
    ```
3.  **Create `names.json`:** Create a `names.json` file in the same directory:
    ```json
    {
        "Ayşe": "عائشة",
        "Fatma": "فاطمة",
        // ... add all names from the provided names.json ...
    }
    ```
4.  **Install FFmpeg:** Make sure FFmpeg is installed on your system and accessible via your PATH. For Ubuntu/Debian:
    ```bash
    sudo apt update
    sudo apt install ffmpeg
    ```
    For Windows/macOS, refer to the [official FFmpeg guide](https://ffmpeg.org/download.html).
5.  **Install Python Dependencies:**
    ```bash
    pip install -r requirements.txt
    pip install transformers accelerate torch optimum auto-gptq ctranslate pyrogram ytdl-patched ffmpeg-python validators srt
    ```
    *Note: The `faster_whisper` wheel might need manual download and installation if `pip install -r` fails for the direct URL.*
6.  **Run the Bot:**
    ```bash
    python main.py
    ```

## ⚙️ Workflow

1.  **User Sends Link:** User sends a video URL (YouTube, Twitter, etc.) to the bot.
2.  **Video Info:** The bot fetches video details (title, duration, size) and presents them with an inline "🚀 Start Translation" button.
3.  **Processing:** Upon button press, the bot sends a "Processing..." message with simple text updates (e.g., "Downloading ⏳" -> "Extracting Audio 🎙" -> "Translating 🌍" -> "Hardsubbing 🎞").
4.  **Delivery:** Once complete, the bot sends the translated video (hardsubbed) and the generated `.srt` file as a document.
5.  **Cleanup:** Temporary files are automatically deleted after sending.

## ⚠️ Important Notes

-   **Personal Use Only:** This bot is strictly for single-user personal use. No databases, multi-user systems, admin panels, or per-user settings are implemented.
-   **Font for Hardsub:** The `processor.py` attempts to use `NotoSansArabic-Regular.ttf` for hardsubbing. Ensure this font or a suitable alternative is available in your environment (`/usr/share/fonts/truetype/noto/` on many Linux systems, including Colab). If not found, FFmpeg will fall back to a generic font.
-   **GPU for Performance:** Using a GPU (`torch.cuda.is_available()`) will significantly speed up Faster-Whisper and Qwen model inference.


Writing README.md
